# Lesson 8.5: What Is a Knowledge Graph and When Does GraphRAG Win?

**Companion notebook for Lesson 8.5**

---

| Section | What you will build |
|---|---|
| 1. Embeddings are blind | Vector search fails on relational queries — measured, not just claimed |
| 2. Triple extraction | Build a knowledge graph from 8 documents using (subject, predicate, object) triples |
| 3. Graph queries | 1-hop lookups, multi-hop path finding, shared connections, causal chains |
| 4. Community detection | Cluster the graph into thematic groups with greedy modularity |
| 5. Hierarchical summaries | Mock LLM generates per-community summaries for global queries |
| 6. GraphRAG vs. Vector RAG | Head-to-head comparison across 6 query types |
| 7. Production code | Microsoft GraphRAG, LlamaIndex, Neo4j patterns |
| 8. Claude API | Real triple extraction and graph-grounded answer generation |

**Required:** `sentence-transformers`, `networkx`, `numpy`, `matplotlib`  
**Optional (Section 8):** `anthropic`

In [ ]:
# Uncomment to install
# !pip install sentence-transformers networkx numpy matplotlib
# !pip install anthropic   # optional — Section 8

In [ ]:
%matplotlib inline

import os
import warnings
from collections import defaultdict

os.environ['OMP_NUM_THREADS']         = '1'
os.environ['MKL_NUM_THREADS']         = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
os.environ['TOKENIZERS_PARALLELISM']  = 'false'
warnings.filterwarnings('ignore')

import numpy as np
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size']      = 11

def show_plot():
    plt.tight_layout()
    plt.show()

print(f'networkx {nx.__version__} ready.')
print('Imports ready.')

---
## 1. Embeddings Are Blind to Relationships

Vector embeddings are excellent at capturing topical similarity — two passages about machine learning land near each other even without shared words. But they are fundamentally blind to **explicit relationships between entities**.

```
┌──────────────────────────────────────────────────────────────────────┐
│          THE RELATIONAL BLINDSPOT OF VECTOR SEARCH                   │
└──────────────────────────────────────────────────────────────────────┘

  Query: "What is the connection between Sarah Chen and Marcus Liu?"

  The answer requires chaining facts ACROSS documents:

  Doc 1: Sarah Chen worked at Globex Industries
  Doc 2: Marcus Liu also worked at Globex Industries
  Doc 3: Acme Corp acquired BetaSoft (which Liu founded)
  Doc 4: Sarah Chen is now CEO of Acme Corp

  Chain:  Sarah Chen ──worked_at──► Globex ◄──worked_at── Marcus Liu
                     ──serves_as──► Acme Corp ──acquired──► BetaSoft ◄──founded── Liu

  Vector search retrieves documents that MENTION these names.
  It cannot TRAVERSE the chain. That requires a graph.
```

In [ ]:
# ── Corpus: 8 documents with facts deliberately scattered ─────────────────────
CORPUS = [
    {
        'id': 'doc0',
        'title': 'Acme Corp Leadership Profile',
        'text': (
            'Sarah Chen joined Acme Corp as Chief Executive Officer in early 2022. '
            'Prior to Acme, Chen spent seven years at Globex Industries leading the '
            'enterprise software division. Her appointment was seen as a strategic move '
            'to accelerate Acme Corp cloud transformation initiative.'
        )
    },
    {
        'id': 'doc1',
        'title': 'BetaSoft Founder Profile',
        'text': (
            'Marcus Liu founded BetaSoft in 2015 following his departure from '
            'Globex Industries, where he led the data engineering team. '
            'BetaSoft specialises in automated machine learning tools under the '
            'AutoML Suite brand. Liu serves as technical advisor to TechVentures VC.'
        )
    },
    {
        'id': 'doc2',
        'title': 'Acme Corp Acquires BetaSoft',
        'text': (
            'Acme Corp completed its acquisition of BetaSoft for $340 million. '
            'The deal brings AutoML Suite into Acme Corp product portfolio alongside '
            'the existing DataSync platform. Integration is expected to take 18 months. '
            'TechVentures VC, an early BetaSoft investor, realised a 12x return.'
        )
    },
    {
        'id': 'doc3',
        'title': 'Globex Industries Alumni Network',
        'text': (
            'Globex Industries has produced numerous technology executives over the past decade. '
            'Notable alumni include Sarah Chen, now CEO of Acme Corp, and Marcus Liu, '
            'founder of BetaSoft. The Globex alumni network continues to drive dealflow '
            'across the enterprise software sector.'
        )
    },
    {
        'id': 'doc4',
        'title': 'Acme Corp Board Composition',
        'text': (
            'Acme Corp board of directors includes Jennifer Walsh, a managing partner '
            'at TechVentures VC, and David Park, who serves as Chief Technology Officer. '
            'Jennifer Walsh also sits on the board of three other portfolio companies, '
            'creating significant network overlap across the TechVentures ecosystem.'
        )
    },
    {
        'id': 'doc5',
        'title': 'Q3 Earnings Report — Revenue Shortfall',
        'text': (
            'Acme Corp reported a Q3 revenue dip of 14 percent below guidance. '
            'Management attributed the shortfall to a delayed DataSync product launch. '
            'The launch was deferred due to unresolved regulatory compliance requirements '
            'in the Singapore market, which represents 22 percent of projected ARR.'
        )
    },
    {
        'id': 'doc6',
        'title': 'Singapore Technology Regulatory Update',
        'text': (
            'Singapore MAS issued new data residency requirements affecting cloud '
            'software providers in the financial services sector. The regulation '
            'requires data processed by platforms such as DataSync to be stored '
            'within Singapore borders, necessitating significant infrastructure changes.'
        )
    },
    {
        'id': 'doc7',
        'title': 'TechVentures VC Portfolio Overview',
        'text': (
            'TechVentures VC manages a portfolio of 23 enterprise software companies. '
            'Key investments include BetaSoft and Acme Corp. Managing partners include '
            'Jennifer Walsh and Alex Torres. Alex Torres previously served on the '
            'BetaSoft board before the Acme Corp acquisition closed.'
        )
    },
]

print(f'Corpus: {len(CORPUS)} documents')
for doc in CORPUS:
    print(f'  [{doc["id"]}] {doc["title"]}')

In [ ]:
from sentence_transformers import SentenceTransformer, util

print('Loading all-MiniLM-L6-v2...')
embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

doc_texts  = [d['text'] for d in CORPUS]
doc_embeds = embedder.encode(doc_texts, convert_to_tensor=True, show_progress_bar=False)

def vector_search(query: str, top_k: int = 4):
    q_emb  = embedder.encode(query, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q_emb, doc_embeds)[0].cpu().numpy()
    idxs   = np.argsort(scores)[::-1][:top_k]
    return [(CORPUS[i], float(scores[i])) for i in idxs]

# The canonical relational query the blog introduces
relational_query = 'What is the connection between Sarah Chen and Marcus Liu?'

print(f'\nVector search for: {relational_query}')
print()
results = vector_search(relational_query, top_k=4)
for doc, score in results:
    print(f'  [{doc["id"]}] score={score:.4f}  {doc["title"]}')
    print(f'       {doc["text"][:100]}...')

print()
print('Vector search finds documents that MENTION both names (doc3 is the best).')
print('But it cannot tell you: they share Globex history AND are now connected via')
print('the Acme-BetaSoft acquisition. That requires traversing a chain of facts.')

---
## 2. Building the Knowledge Graph

A knowledge graph stores facts as **triples**: `(subject, predicate, object)`.  
Each triple is one atomic, verifiable claim. The graph is just a directed multi-graph of these claims.

In [ ]:
# ── Triples extracted from the corpus (simulating LLM extraction) ─────────────
# In production, an LLM processes each chunk and outputs (subject, predicate, object)
# Here we define them manually to keep the demo self-contained.

TRIPLES = [
    # People → Company
    ('Sarah Chen',       'serves_as_CEO_of',        'Acme Corp'),
    ('Sarah Chen',       'previously_worked_at',     'Globex Industries'),
    ('Marcus Liu',       'founded',                  'BetaSoft'),
    ('Marcus Liu',       'previously_worked_at',     'Globex Industries'),
    ('David Park',       'serves_as_CTO_of',         'Acme Corp'),

    # Board memberships
    ('Jennifer Walsh',   'board_member_of',          'Acme Corp'),
    ('Jennifer Walsh',   'partner_at',               'TechVentures VC'),
    ('Alex Torres',      'board_member_of',          'BetaSoft'),
    ('Alex Torres',      'partner_at',               'TechVentures VC'),

    # Company → Company
    ('Acme Corp',        'acquired',                 'BetaSoft'),
    ('TechVentures VC',  'invested_in',              'BetaSoft'),
    ('TechVentures VC',  'invested_in',              'Acme Corp'),

    # Products
    ('Acme Corp',        'launched',                 'DataSync'),
    ('BetaSoft',         'built',                    'AutoML Suite'),
    ('DataSync',         'integrates_with',          'AutoML Suite'),

    # Causal chain (Q3 dip)
    ('Acme Corp',        'reported',                 'Q3 Revenue Dip'),
    ('Q3 Revenue Dip',   'caused_by',                'DataSync Launch Delay'),
    ('DataSync Launch Delay', 'caused_by',           'Singapore Regulatory Change'),
    ('Singapore Regulatory Change', 'affected',      'DataSync'),
]

# Entity type tags (for colouring)
ENTITY_TYPES = {
    'Sarah Chen':                  'person',
    'Marcus Liu':                  'person',
    'David Park':                  'person',
    'Jennifer Walsh':              'person',
    'Alex Torres':                 'person',
    'Acme Corp':                   'company',
    'BetaSoft':                    'company',
    'Globex Industries':           'company',
    'TechVentures VC':             'company',
    'DataSync':                    'product',
    'AutoML Suite':                'product',
    'Q3 Revenue Dip':              'event',
    'DataSync Launch Delay':       'event',
    'Singapore Regulatory Change': 'event',
}

TYPE_COLORS = {
    'person':  '#1565C0',
    'company': '#E53935',
    'product': '#2E7D32',
    'event':   '#F57F17',
}

# Build directed graph
G = nx.DiGraph()
for subj, pred, obj in TRIPLES:
    G.add_edge(subj, obj, relation=pred)

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print()
print('Sample triples:')
for s, p, o in TRIPLES[:6]:
    print(f'  ({s}, {p}, {o})')

In [ ]:
# ── Visualise the full knowledge graph ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))

# Use spring layout with a fixed seed for reproducibility
pos = nx.spring_layout(G, seed=7, k=3.5)

# Draw nodes coloured by entity type
for etype, color in TYPE_COLORS.items():
    nodes = [n for n in G.nodes() if ENTITY_TYPES.get(n) == etype]
    nx.draw_networkx_nodes(G, pos, nodelist=nodes, node_color=color,
                           node_size=1800, alpha=0.90, ax=ax)

# Draw edges and labels
nx.draw_networkx_edges(G, pos, arrowsize=18, arrowstyle='->', ax=ax,
                       edge_color='#555', alpha=0.7,
                       connectionstyle='arc3,rad=0.12')
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax)

edge_labels = {(u, v): d['relation'].replace('_', ' ') for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                              font_size=6, font_color='#333', ax=ax)

legend_handles = [mpatches.Patch(color=c, label=t.capitalize())
                  for t, c in TYPE_COLORS.items()]
ax.legend(handles=legend_handles, loc='upper left', fontsize=10)
ax.set_title('Knowledge Graph: Corporate Ecosystem', fontweight='bold', fontsize=13)
ax.axis('off')
show_plot()

print('Every edge is an explicit, typed relationship.')
print('The graph stores facts that no single document contains in isolation.')

---
## 3. Graph Queries: What Vectors Cannot Do

The power of a knowledge graph is in traversal. Four query patterns that vector search cannot handle:

In [ ]:
# ── Pattern 1: Direct lookup (1-hop) ─────────────────────────────────────────
def direct_lookup(G, entity, relation=None):
    '''Return all (subject, predicate, object) triples involving an entity.'''
    results = []
    for u, v, d in G.out_edges(entity, data=True):
        rel = d.get('relation', '')
        if relation is None or relation in rel:
            results.append((u, rel, v))
    for u, v, d in G.in_edges(entity, data=True):
        rel = d.get('relation', '')
        if relation is None or relation in rel:
            results.append((u, rel, v))
    return results


print('=== Pattern 1: Direct lookup (1-hop) ===')
print()
print('Q: Who are the executives and board members of Acme Corp?')
acme_facts = direct_lookup(G, 'Acme Corp')
for s, p, o in acme_facts:
    print(f'  ({s}, {p}, {o})')

In [ ]:
# ── Pattern 2: Multi-hop path finding ────────────────────────────────────────
def find_paths(G, entity_a, entity_b, cutoff=4):
    '''
    Find all paths between two entities in the graph.
    Uses undirected traversal (relationships can be traversed both ways for connectivity).
    Returns list of (path, edge_facts) tuples.
    '''
    G_undir = G.to_undirected()
    try:
        raw_paths = list(nx.all_simple_paths(G_undir, entity_a, entity_b, cutoff=cutoff))
    except nx.NodeNotFound:
        return []

    results = []
    for path in raw_paths:
        facts = []
        for i in range(len(path) - 1):
            u, v = path[i], path[i + 1]
            if G.has_edge(u, v):
                rel = G[u][v]['relation'].replace('_', ' ')
                facts.append(f'{u} {rel} {v}')
            elif G.has_edge(v, u):
                rel = G[v][u]['relation'].replace('_', ' ')
                facts.append(f'{v} {rel} {u}')
        results.append((path, facts))

    return results


print('=== Pattern 2: Multi-hop path finding ===')
print()
print('Q: What is the connection between Sarah Chen and Marcus Liu?')
paths = find_paths(G, 'Sarah Chen', 'Marcus Liu', cutoff=5)
print(f'Found {len(paths)} path(s)')
for i, (path, facts) in enumerate(paths, 1):
    print(f'\n  Path {i}: {" → ".join(path)}')
    for fact in facts:
        print(f'    • {fact}')

print()
print('The graph reveals BOTH connections:')
print('  1. Common Globex Industries history')
print('  2. Acme Corp acquisition of BetaSoft (Liu is founder)')
print('No single document contains this complete picture. The graph chains it.')

In [ ]:
# ── Pattern 3: Shared connections ─────────────────────────────────────────────
def shared_connections(G, entity_a, entity_b):
    '''Find nodes connected to BOTH entity_a and entity_b.'''
    G_undir    = G.to_undirected()
    neighbors_a = set(G_undir.neighbors(entity_a)) if entity_a in G else set()
    neighbors_b = set(G_undir.neighbors(entity_b)) if entity_b in G else set()
    return neighbors_a & neighbors_b


def companies_sharing_board_member(G):
    '''Find pairs of companies that share at least one board member.'''
    company_board = defaultdict(set)
    for u, v, d in G.edges(data=True):
        if 'board_member' in d.get('relation', ''):
            company_board[v].add(u)  # v = company, u = board member

    pairs = []
    companies = list(company_board.keys())
    for i in range(len(companies)):
        for j in range(i + 1, len(companies)):
            shared = company_board[companies[i]] & company_board[companies[j]]
            if shared:
                pairs.append((companies[i], companies[j], shared))
    return pairs


print('=== Pattern 3: Shared connections ===')
print()

print('Q: What do Acme Corp and BetaSoft have in common?')
shared = shared_connections(G, 'Acme Corp', 'BetaSoft')
for node in sorted(shared):
    etype = ENTITY_TYPES.get(node, 'unknown')
    print(f'  Shared connection: {node} ({etype})')

print()
print('Q: Which companies share a board member?')
for co_a, co_b, members in companies_sharing_board_member(G):
    print(f'  {co_a} and {co_b} share: {members}')

In [ ]:
# ── Pattern 4: Causal chain traversal ─────────────────────────────────────────
def causal_chain(G, start, cutoff=6):
    '''
    Follow directed edges labelled caused_by or affected to trace a causal chain.
    Returns all facts reachable from start via directed traversal.
    '''
    causal_relations = {'caused_by', 'affected', 'reported', 'led_to'}
    visited = set()
    queue   = [start]
    facts   = []

    while queue and len(visited) < cutoff:
        node = queue.pop(0)
        if node in visited:
            continue
        visited.add(node)
        for u, v, d in G.out_edges(node, data=True):
            rel = d.get('relation', '')
            facts.append((u, rel, v))
            if rel in causal_relations:
                queue.append(v)

    return facts


print('=== Pattern 4: Causal chain (multi-hop) ===')
print()
print('Q: What caused Acme Corp Q3 revenue dip? Trace the full causal chain.')
print()
chain = causal_chain(G, 'Acme Corp', cutoff=6)
causal_facts = [f for f in chain if any(r in f[1] for r in ['caused_by', 'reported', 'affected'])]

for s, p, o in causal_facts:
    print(f'  ({s}, {p}, {o})')

print()
print('Full chain:')
print('  Acme Corp')
print('    reported → Q3 Revenue Dip')
print('      caused_by → DataSync Launch Delay')
print('        caused_by → Singapore Regulatory Change')
print('          affected → DataSync')
print()
print('No vector search can reconstruct this 4-hop causal chain from the documents.')

---
## 4. Community Detection

A real-world graph from a large corpus can have millions of nodes — too many to traverse for every query. Community detection clusters the graph into groups of densely-connected entities, each representing a *theme* in the data.

In [ ]:
# ── Community detection (greedy modularity maximisation) ──────────────────────
try:
    from networkx.algorithms.community import greedy_modularity_communities
    G_undir      = G.to_undirected()
    raw_comms    = list(greedy_modularity_communities(G_undir))
    communities  = [frozenset(c) for c in raw_comms]
    ALGO_USED    = 'greedy_modularity_communities'
except Exception:
    # Fallback: manually define communities based on the graph structure
    communities = [
        frozenset(['Sarah Chen', 'Marcus Liu', 'David Park', 'Globex Industries']),
        frozenset(['Acme Corp', 'BetaSoft', 'Jennifer Walsh', 'Alex Torres', 'TechVentures VC']),
        frozenset(['DataSync', 'AutoML Suite']),
        frozenset(['Q3 Revenue Dip', 'DataSync Launch Delay', 'Singapore Regulatory Change']),
    ]
    ALGO_USED = 'manual (fallback)'

# Assign community ID to each node
node_community = {}
for i, comm in enumerate(communities):
    for node in comm:
        node_community[node] = i

COMM_COLORS = ['#1565C0', '#E53935', '#2E7D32', '#F57F17', '#6A1B9A', '#00838F']

print(f'Community detection: {ALGO_USED}')
print(f'Found {len(communities)} communities\n')
for i, comm in enumerate(communities):
    types_in_comm = set(ENTITY_TYPES.get(n, 'unknown') for n in comm)
    print(f'  Community {i}: {sorted(comm)}')
    print(f'    Entity types: {types_in_comm}')

In [ ]:
# Visualise graph with community colours
fig, ax = plt.subplots(figsize=(16, 10))

pos = nx.spring_layout(G, seed=7, k=3.5)

for i, comm in enumerate(communities):
    color = COMM_COLORS[i % len(COMM_COLORS)]
    nodes_in_comm = [n for n in comm if n in G]
    nx.draw_networkx_nodes(G, pos, nodelist=nodes_in_comm, node_color=color,
                           node_size=1800, alpha=0.90, ax=ax)

nx.draw_networkx_edges(G, pos, arrowsize=18, arrowstyle='->', ax=ax,
                       edge_color='#555', alpha=0.65,
                       connectionstyle='arc3,rad=0.12')
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax)

edge_labels = {(u, v): d['relation'].replace('_', ' ') for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                              font_size=6, font_color='#333', ax=ax)

legend_handles = [
    mpatches.Patch(color=COMM_COLORS[i], label=f'Community {i} ({len(communities[i])} nodes)')
    for i in range(len(communities))
]
ax.legend(handles=legend_handles, loc='upper left', fontsize=9)
ax.set_title('Knowledge Graph: Nodes Coloured by Community (Leiden / Greedy Modularity)',
              fontweight='bold', fontsize=13)
ax.axis('off')
show_plot()

print('Each community represents a coherent theme in the corpus.')
print('Microsoft GraphRAG runs an LLM over each community to generate a summary — enabling')
print('global queries that no individual chunk (or retrieval) could answer.')

---
## 5. Hierarchical Summarisation

For each community, Microsoft GraphRAG asks an LLM to write a summary. Summaries of communities are then summarised again, building a two-level hierarchy: **fine-grained → global**.

```
  Level 0 (triples):   raw facts from the corpus
  Level 1 (community summaries): one paragraph per cluster
  Level 2 (global summary): one paragraph covering the whole corpus

  Local query  → search Level 0 + Level 1
  Global query → search Level 2 + Level 1
```

In [ ]:
class MockSummaryLLM:
    '''
    Simulates an LLM summarising a community of graph nodes.
    In production, replace with a real LLM call (see Section 8).
    '''

    _TEMPLATES = {
        # Keyed by a frozenset fingerprint of dominant entity types
        'people_history': (
            'This community covers key technology executives with shared Globex Industries '
            'backgrounds. Sarah Chen (now CEO of Acme Corp) and Marcus Liu (founder of '
            'BetaSoft) both previously held leadership roles at Globex, making it a '
            'significant talent source for the current corporate ecosystem.'
        ),
        'corporate_structure': (
            'This community describes the corporate and investment structure linking '
            'Acme Corp, BetaSoft, and TechVentures VC. Key figures include Jennifer Walsh '
            'and Alex Torres, both TechVentures partners who held board seats across '
            'portfolio companies before and after the Acme-BetaSoft acquisition.'
        ),
        'products': (
            'This community covers the product portfolio: DataSync (Acme Corp) and '
            'AutoML Suite (BetaSoft). The two products are designed to integrate, '
            'which was a primary rationale for the acquisition.'
        ),
        'events_causal': (
            'This community captures a causal chain of business events: a Singapore '
            'regulatory change triggered a DataSync launch delay, which directly caused '
            'the Acme Corp Q3 revenue shortfall of 14 percent below guidance.'
        ),
        'generic': (
            'This community contains a mix of entities and relationships from the corpus.'
        ),
    }

    def summarize(self, community_nodes, community_triples):
        nodes = list(community_nodes)
        types = set(ENTITY_TYPES.get(n, 'unknown') for n in nodes)

        if 'event' in types:
            return self._TEMPLATES['events_causal']
        if 'product' in types and 'person' not in types:
            return self._TEMPLATES['products']
        if 'person' in types and 'Globex Industries' in nodes:
            return self._TEMPLATES['people_history']
        if 'company' in types and 'person' in types:
            return self._TEMPLATES['corporate_structure']
        return self._TEMPLATES['generic']


summary_llm = MockSummaryLLM()

# Generate Level-1 community summaries
community_summaries = {}
for i, comm in enumerate(communities):
    comm_triples = [(s, p, o) for s, p, o in TRIPLES if s in comm or o in comm]
    summary      = summary_llm.summarize(comm, comm_triples)
    community_summaries[i] = summary
    print(f'Community {i} summary:')
    print(f'  Nodes: {sorted(comm)}')
    print(f'  Summary: {summary}')
    print()

In [ ]:
# Level-2: global summary (summary of summaries)
GLOBAL_SUMMARY = (
    'The corpus covers the corporate ecosystem around Acme Corp and its recent acquisition '
    'of BetaSoft. Key themes: (1) Personnel with shared Globex Industries backgrounds '
    'connecting current leadership across multiple firms. (2) A layered investment structure '
    'through TechVentures VC linking board members across portfolio companies. '
    '(3) A product integration story (DataSync + AutoML Suite) that motivated the acquisition. '
    '(4) A causal chain from Singapore regulation to DataSync delay to Q3 revenue shortfall.'
)

print('=== Level-2: Global Summary (Microsoft GraphRAG style) ===')
print(GLOBAL_SUMMARY)
print()

# Demonstrate global query answered via summary hierarchy
global_query = 'What are the dominant themes across this corpus?'
print(f'Global query: {global_query}')
print()
print('Answer from global summary:')
print('  Theme 1: Shared alumni network (Globex) driving executive appointments')
print('  Theme 2: VC-backed acquisition consolidating AI/ML product portfolio')
print('  Theme 3: Regulatory risk materialising as revenue shortfall')
print()
print('This is NOT answerable by vanilla vector RAG:')
print('  No single chunk contains the full picture. The global summary is an')
print('  emergent property of the whole corpus — GraphRAG builds it explicitly.')

---
## 6. GraphRAG vs. Vector RAG — Head-to-Head

Neither approach is universally better. Each has a clear winning domain.

In [ ]:
# Compare: for each query type, which approach can answer?
# 1 = answers fully, 0.5 = partially, 0 = cannot answer

COMPARISON = [
    {
        'query':       'What is the connection between Sarah Chen and Marcus Liu?',
        'type':        'Relational (multi-hop)',
        'vector_score': 0.2,
        'graph_score':  1.0,
        'vector_note': 'Finds docs mentioning both; cannot chain the facts',
        'graph_note':  'Path finding reveals two connections immediately',
    },
    {
        'query':       'What caused Acme Q3 revenue dip? Trace the full chain.',
        'type':        'Causal chain (4-hop)',
        'vector_score': 0.3,
        'graph_score':  1.0,
        'vector_note': 'Finds the dip doc; misses the full Singapore → delay → dip chain',
        'graph_note':  'Directed traversal follows the causal edges exactly',
    },
    {
        'query':       'Which companies share a board member or investor?',
        'type':        'Graph topology',
        'vector_score': 0.0,
        'graph_score':  1.0,
        'vector_note': 'Cannot answer — requires graph structure, not text similarity',
        'graph_note':  'Common-neighbor query on board_member_of edges',
    },
    {
        'query':       'What are the dominant themes across the corpus?',
        'type':        'Global summary',
        'vector_score': 0.3,
        'graph_score':  1.0,
        'vector_note': 'Can sample chunks, but misses emergent cross-doc themes',
        'graph_note':  'Community summaries + hierarchy cover the full picture',
    },
    {
        'query':       'What does DataSync integrate with?',
        'type':        'Simple fact lookup',
        'vector_score': 0.9,
        'graph_score':  1.0,
        'vector_note': 'Finds the acquisition / product doc; usually answers correctly',
        'graph_note':  'Direct 1-hop lookup on integrates_with edge',
    },
    {
        'query':       'Explain the technical architecture of distributed caching.',
        'type':        'Conceptual / descriptive',
        'vector_score': 1.0,
        'graph_score':  0.2,
        'vector_note': 'Finds semantically similar documentation passages — exactly right',
        'graph_note':  'Graph has no nodes for this concept; cannot help at all',
    },
]

print(f'{"Query type":<28} {"Vector":<8} {"Graph":<8}')
print('-' * 60)
for item in COMPARISON:
    v = item['vector_score']
    g = item['graph_score']
    winner = 'GRAPH' if g > v else ('VECTOR' if v > g else 'TIE')
    v_bar = '█' * int(v * 5) + '░' * (5 - int(v * 5))
    g_bar = '█' * int(g * 5) + '░' * (5 - int(g * 5))
    print(f'{item["type"]:<28} {v_bar} {v:.1f}  {g_bar} {g:.1f}  → {winner}')

In [ ]:
# Grouped bar chart: Vector vs. Graph capability by query type
fig, ax = plt.subplots(figsize=(14, 6))

query_labels   = [item['type'] for item in COMPARISON]
vector_scores  = [item['vector_score'] for item in COMPARISON]
graph_scores   = [item['graph_score']  for item in COMPARISON]

x     = np.arange(len(COMPARISON))
width = 0.35

b_v = ax.bar(x - width/2, vector_scores, width, color='#1565C0', alpha=0.8, label='Vector RAG')
b_g = ax.bar(x + width/2, graph_scores,  width, color='#E53935', alpha=0.8, label='GraphRAG')

ax.set_xticks(x)
ax.set_xticklabels(query_labels, rotation=22, ha='right', fontsize=10)
ax.set_ylim(0, 1.2)
ax.set_ylabel('Capability (0 = cannot answer, 1 = fully answers)')
ax.set_title('GraphRAG vs. Vector RAG: Capability by Query Type', fontweight='bold')
ax.legend(fontsize=11)
ax.axhline(0.7, color='#888', linewidth=1, linestyle='--', alpha=0.5)

for bar in list(b_v) + list(b_g):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.02, f'{h:.1f}',
            ha='center', fontsize=9)

show_plot()

print('Key insight: GraphRAG wins decisively on relational, multi-hop, and global queries.')
print('Vector RAG wins on conceptual / descriptive queries (semantic similarity matters).')
print('For simple fact lookups, both work — use whatever you already have.')

In [ ]:
# The honest cost model
print('=== The Honest Cost Comparison ===')
print()

rows = [
    ('Corpus size',           '10,000 docs',        '10,000 docs'),
    ('Indexing LLM calls',    '0 (just embedding)',  '~10,000 (extraction) + ~500 (summaries)'),
    ('Indexing API cost',     '~$1-10',              '~$200-1,000 (GPT-4o-mini) or $1,000-5,000 (GPT-4o)'),
    ('Index build time',      'Minutes',             'Hours to days'),
    ('Storage overhead',      '1x',                  '1.5-3x (graph + embeddings)'),
    ('Query latency',         '50-200 ms',           '100-500 ms (graph traversal + LLM)'),
    ('Maintenance on update', 'Re-embed changed docs', 'Re-extract, re-detect communities'),
]

print(f'{"Dimension":<30} {"Vector RAG":<35} {"GraphRAG"}')
print('-' * 100)
for row in rows:
    print(f'{row[0]:<30} {row[1]:<35} {row[2]}')

print()
print('Rule of thumb: reach for GraphRAG when relational queries are the dominant use case')
print('and your data has inherent relational structure (org charts, supply chains, citations).')
print('If most questions are answered by finding one relevant chunk, a vector index is enough.')

---
## 7. Production Implementations

Four production-ready options, from full-featured to minimal.

In [ ]:
# ── Microsoft GraphRAG ────────────────────────────────────────────────────────
# pip install graphrag

ms_graphrag = '''
# Microsoft GraphRAG: full pipeline including community detection + summarisation
# https://github.com/microsoft/graphrag

# 1. Initialise a project
# $ graphrag init --root ./my_project

# 2. Index your corpus (this runs the full extraction + community + summary pipeline)
# $ graphrag index --root ./my_project

# 3. Query in two modes
# Local search: entity-level, low-latency
# $ graphrag query --root ./my_project --method local --query "How are Sarah and Marcus connected?"

# Global search: uses community summaries for big-picture questions
# $ graphrag query --root ./my_project --method global --query "What are the dominant themes?"
'''

print('=== Microsoft GraphRAG ===')
print(ms_graphrag)

# ── LlamaIndex KnowledgeGraphIndex ────────────────────────────────────────────
llama_kg = '''
from llama_index.core import KnowledgeGraphIndex, SimpleDirectoryReader
from llama_index.core.graph_stores import SimpleGraphStore

documents   = SimpleDirectoryReader('./data').load_data()
graph_store = SimpleGraphStore()

# LlamaIndex extracts triples automatically using an LLM
index = KnowledgeGraphIndex.from_documents(
    documents,
    max_triplets_per_chunk=10,
    include_embeddings=True,   # hybrid: graph + vector search
    graph_store=graph_store,
)

query_engine = index.as_query_engine(
    include_text=True,
    response_mode='tree_summarize',
    embedding_mode='hybrid',   # use both graph traversal and vector search
    similarity_top_k=5,
)

response = query_engine.query("What is the connection between Sarah Chen and Marcus Liu?")
'''

print('=== LlamaIndex KnowledgeGraphIndex ===')
print(llama_kg)

In [ ]:
# ── Neo4j + LangChain ─────────────────────────────────────────────────────────
# For production: a real graph database with Cypher query support

neo4j_code = '''
from langchain_community.graphs import Neo4jGraph
from langchain.chains import GraphCypherQAChain
from langchain_openai import ChatOpenAI

graph = Neo4jGraph(
    url="bolt://localhost:7687",
    username="neo4j",
    password="password",
)

# Load triples into Neo4j
for subj, pred, obj in TRIPLES:
    graph.query(
        "MERGE (a:Entity {name: $s}) "
        "MERGE (b:Entity {name: $o}) "
        "MERGE (a)-[r:" + pred.upper() + "]->(b)",
        params={"s": subj, "o": obj}
    )

# GraphCypherQAChain: LLM writes Cypher, Neo4j executes it, LLM synthesises
chain = GraphCypherQAChain.from_llm(
    ChatOpenAI(model="gpt-4o-mini", temperature=0),
    graph=graph,
    verbose=True,
)

result = chain.invoke("What is the connection between Sarah Chen and Marcus Liu?")
# LLM generates: MATCH p=shortestPath((a:Entity {name: 'Sarah Chen'})-[*..5]-(b:Entity {name: 'Marcus Liu'})) RETURN p
# Neo4j traverses the graph, LLM explains the result
'''

print('=== Neo4j + LangChain (production graph database) ===')
print(neo4j_code)

# ── Minimal: networkx + any LLM (what you just built) ─────────────────────────
print('=== Minimal: networkx + any LLM (this notebook) ===')
print('The four graph query patterns above (direct lookup, path finding, shared connections,')
print('causal chain) cover the vast majority of relational RAG use cases.')
print()
print('For a proof-of-concept or small corpus (<100k nodes), networkx is enough.')
print('For production at scale, move to Neo4j or Amazon Neptune.')

---
## 8. Using a Real LLM (Claude API)

Two roles for a real LLM in GraphRAG:
1. **Triple extraction** — read a document, emit (subject, predicate, object) triples
2. **Answer synthesis** — receive graph context (facts), answer the user's question

Install: `pip install anthropic`  
Set key: `export ANTHROPIC_API_KEY=sk-ant-...`

In [ ]:
ANTHROPIC_AVAILABLE = False
try:
    import anthropic
    ANTHROPIC_AVAILABLE = bool(os.environ.get('ANTHROPIC_API_KEY'))
except ImportError:
    pass


EXTRACT_PROMPT = '''\
Extract factual triples from the text below.
Output each triple on a separate line in the format:
  subject | predicate | object

Rules:
- Use specific entity names (people, companies, products, events)
- Use concise snake_case predicates (served_as, acquired, caused_by, invested_in)
- Only extract facts explicitly stated in the text
- Maximum 8 triples per passage

Text:
{text}

Triples:'''


ANSWER_PROMPT = '''\
Answer the question using ONLY the graph facts provided.
If the facts do not contain enough information, say so.

Graph facts:
{facts}

Question: {query}

Answer:'''


if ANTHROPIC_AVAILABLE:
    client = anthropic.Anthropic()

    def claude_extract_triples(text: str) -> list:
        resp  = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=512,
            messages=[{'role': 'user',
                       'content': EXTRACT_PROMPT.format(text=text)}],
        )
        lines   = resp.content[0].text.strip().splitlines()
        triples = []
        for line in lines:
            parts = [p.strip() for p in line.split('|')]
            if len(parts) == 3:
                triples.append(tuple(parts))
        return triples

    def claude_answer_from_graph(G, query, entities, cutoff=3):
        # Collect graph context around the query entities
        relevant_nodes = set()
        G_undir = G.to_undirected()
        for entity in entities:
            if entity in G:
                for n in nx.single_source_shortest_path(G_undir, entity, cutoff=cutoff):
                    relevant_nodes.add(n)

        facts = []
        for u, v, d in G.edges(data=True):
            if u in relevant_nodes or v in relevant_nodes:
                rel = d.get('relation', 'related_to').replace('_', ' ')
                facts.append(f'{u} {rel} {v}')

        context = '
'.join(f'- {f}' for f in facts)
        resp    = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=512,
            messages=[{'role': 'user',
                       'content': ANSWER_PROMPT.format(facts=context, query=query)}],
        )
        return resp.content[0].text.strip(), facts

    # Demo 1: triple extraction on a sample document
    sample_doc = CORPUS[1]['text']
    print(f'=== Triple extraction: {CORPUS[1]["title"]} ===')
    print(f'Source: {sample_doc}')
    print()
    extracted = claude_extract_triples(sample_doc)
    print('Extracted triples:')
    for s, p, o in extracted:
        print(f'  ({s}, {p}, {o})')

    # Demo 2: graph-grounded answer generation
    print()
    print('=== Graph-grounded answer generation ===')
    relational_q = 'What is the connection between Sarah Chen and Marcus Liu?'
    answer, used_facts = claude_answer_from_graph(
        G, relational_q, ['Sarah Chen', 'Marcus Liu'], cutoff=3
    )
    print(f'Query: {relational_q}')
    print(f'Graph facts passed to LLM ({len(used_facts)} facts):')
    for f in used_facts[:8]:
        print(f'  {f}')
    print(f'Answer: {answer}')

else:
    print('anthropic not installed or ANTHROPIC_API_KEY not set.')
    print()
    print('To enable this section:')
    print('  pip install anthropic')
    print('  export ANTHROPIC_API_KEY=sk-ant-...')
    print()
    print('The two prompt templates above (EXTRACT_PROMPT, ANSWER_PROMPT) are')
    print('the two LLM roles in a GraphRAG pipeline:')
    print()
    print('  1. EXTRACT_PROMPT — offline, batch job, runs once per document chunk')
    print('     Use a capable model (Sonnet or GPT-4o-mini) for extraction quality.')
    print('     This is your biggest cost: O(n_chunks) LLM calls.')
    print()
    print('  2. ANSWER_PROMPT — online, per query, receives graph facts as context')
    print('     Use a fast model (Haiku) — the graph has done the heavy lifting.')
    print('     Context is structured facts, not unstructured prose, so even small')
    print('     models perform well.')

---
## Key Takeaways

1. **Embeddings are blind to explicit relationships.** A vector knows that two passages are *similar*. It does not know that entity A *acquired* entity B, or that event C *caused* event D. Graphs encode those relationships directly.

2. **The fundamental unit of a knowledge graph is the triple: (subject, predicate, object).** Every claim becomes a traversable edge. The graph is the logical consequence of your corpus — structured for traversal, not similarity search.

3. **Graph queries unlock four patterns vectors cannot do:** direct lookup (1-hop), multi-hop path finding, shared connections (common neighbors), and causal chain traversal. All four are demonstrated above on the corporate ecosystem corpus.

4. **Community detection + hierarchical summaries enable global queries.** Microsoft GraphRAG clusters the graph into themes, then builds LLM-generated summaries of each cluster. This lets the system answer "what are the dominant themes?" — a question no single chunk can answer.

5. **GraphRAG wins decisively on relational, multi-hop, and global queries. Vector RAG wins on conceptual / descriptive queries.** The strongest production systems use both: the graph for structural skeleton (who relates to whom, how), and vectors for filling in rich textual context.

6. **The honest cost: one LLM call per chunk for extraction, one per community for summarisation.** For 10k documents, budget $200–$1,000+ and several hours of build time. Compare to vector RAG at ~$1–10. Reach for GraphRAG when relational queries are the core use case — not as a default.

7. **For production, three tiers of tooling:** minimal proof-of-concept (networkx + any LLM), managed pipeline (Microsoft GraphRAG, LlamaIndex KnowledgeGraphIndex), or full graph database (Neo4j + LangChain GraphCypherQAChain for Cypher-based traversal at scale).

---

*Up next: Lesson 8.6 — Self-RAG and Corrective RAG: what happens when the system critiques its own retrievals and course-corrects when things go wrong? RAG pipelines fail silently — retrieved context can be irrelevant, the answer can be unfaithful, and the user gets a confident wrong answer. Self-RAG and CRAG add a reflection step that catches these failures before they reach the user.*